# Lab Assignment 2 - Part B: k-Nearest Neighbor Classification
Please refer to the `README.pdf` for full laboratory instructions.


## Problem Statement
In this part, you will implement the k-Nearest Neighbor (k-NN) classifier and evaluate it on two datasets:
- **Lenses Dataset**: A small dataset for contact lens prescription
- **Credit Approval (CA) Dataset**: Credit card application data with binary labels (+/-)

### Your Tasks
1. **Preprocess the data**: Handle missing values and normalize features
2. **Implement k-NN** with L2 distance
3. **Evaluate** on both datasets for different values of k
4. **Discuss** your results

### Datasets
The data files are located in the `credit 2017/` folder:
- `lenses.training`, `lenses.testing`
- `crx.data.training`, `crx.data.testing`
- `crx.names` (describes the features)


## Setup


In [2]:
# Library declarations
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter


In [3]:
# Data paths
DATA_PATH = "credit 2017/"

# Load Lenses data
def load_lenses_data():
    """Load the lenses dataset."""
    train_data = np.loadtxt(DATA_PATH + "lenses.training", delimiter=',')
    test_data = np.loadtxt(DATA_PATH + "lenses.testing", delimiter=',')
    
    # First column is ID, last column is label
    X_train = train_data[:, 1:-1]
    y_train = train_data[:, -1]
    X_test = test_data[:, 1:-1]
    y_test = test_data[:, -1]
    
    return X_train, y_train, X_test, y_test

# Load Credit Approval data
def load_credit_data():
    """
    Load the Credit Approval dataset.
    Note: This dataset contains missing values (?) and mixed types.
    You will need to preprocess it.
    """
    # TODO: Implement data loading
    # The data is comma-separated
    # Missing values are marked with '?'
    # Last column is the label ('+' or '-')
    # Load raw data (no preprocessing yet)
    # Load raw data as strings (important for '?')
    train_data = np.loadtxt(DATA_PATH + "crx.data.training",delimiter=",", dtype=str)
    test_data = np.loadtxt( DATA_PATH + "crx.data.testing",delimiter=",",dtype=str)

    # Split features and labels
    X_train = train_data[:, :-1]
    y_train = train_data[:, -1]
    
    X_test = test_data[:, :-1]
    y_test = test_data[:, -1]

    # Convert labels '+' to 1, '-' to 0
    y_train = np.array([1 if label == '+' else 0 for label in y_train])
    y_test = np.array([1 if label == '+' else 0 for label in y_test])

    return X_train, y_train, X_test, y_test
    pass

# Test loading lenses data
X_train_lenses, y_train_lenses, X_test_lenses, y_test_lenses = load_lenses_data()
print(f"Lenses - Train: {X_train_lenses.shape}, Test: {X_test_lenses.shape}")


Lenses - Train: (18, 3), Test: (6, 3)


## Task 1: Data Preprocessing
For the Credit Approval dataset, you need to:
1. **Handle missing values** (marked with '?'):
   - Categorical features: replace with mode/median
   - Numerical features: replace with label-conditioned mean
2. **Normalize features** using z-scaling:
   $$z_i^{(m)} = \frac{x_i^{(m)} - \mu_i}{\sigma_i}$$

Document exactly how you handle each feature!


In [ ]:
def preprocess_credit_data(train_file, test_file):
    """
    Preprocess the Credit Approval dataset.
    
    Steps:
    1. Load the data
    2. Handle missing values
    3. Encode categorical variables
    4. Normalize numerical features
    
    Returns:
    --------
    X_train, y_train, X_test, y_test : numpy arrays
    """
    # TODO: Implement preprocessing
    # Hint: Read crx.names to understand the features
    # Feature types (from crx.names):
    # A1: categorical (b, a)
    # A2: continuous
    # A3: continuous
    # A4: categorical (u, y, l, t)
    # A5: categorical (g, p, gg)
    # A6: categorical (c, d, cc, i, j, k, m, r, q, w, x, e, aa, ff)
    # A7: categorical (v, h, bb, j, n, z, dd, ff, o)
    # A8: continuous
    # A9: categorical (t, f)
    # A10: categorical (t, f)
    # A11: continuous
    # A12: categorical (t, f)
    # A13: categorical (g, p, s)
    # A14: continuous
    # A15: continuous
    # A16: label (+, -)
    train_data = np.loadtxt(DATA_PATH + train_file,delimiter=",", dtype=str)
    test_data = np.loadtxt(DATA_PATH + test_file,delimiter=",", dtype=str)
    
    X_train = train_data[:, :-1]
    y_train = train_data[:, -1]
    
    X_test = test_data[:, :-1]
    y_test = test_data[:, -1]
    
    # handling the output
    
     # Convert labels
    y_train = np.array([1 if y == '+' else 0 for y in y_train])
    y_test  = np.array([1 if y == '+' else 0 for y in y_test])

    # Feature indices
    categorical_cols = [0,3,4,5,6,8,9,11,12]
    continuous_cols  = [1,2,7,10,13,14]
    
     # Data Imputation 
     
    for col in categorical_cols:
        # mode imputation
        values = X_train[X_train[:, col] != '?', col]
        # update missing values to have the mode
        mode = Counter(values).most_common(1)[0][0]
        X_train[X_train[:, col] == '?', col] = mode
        X_test[X_test[:, col] == '?', col] = mode

    for col in continuous_cols:
        for label in [0, 1]:
            mask = (y_train == label) & (X_train[:, col] != '?')
            mean = X_train[mask, col].astype(float).mean()

            replace_mask = (y_train == label) & (X_train[:, col] == '?')
            X_train[replace_mask, col] = mean

        # If test has '?', use overall training mean
        overall_mean = X_train[:, col].astype(float).mean()
        X_test[X_test[:, col] == '?', col] = overall_mean

    #Encoding categorical features
    for col in categorical_cols:
        unique_vals = np.unique(X_train[:, col])
        mapping = {val: idx for idx, val in enumerate(unique_vals)}

        X_train[:, col] = np.array([mapping[v] for v in X_train[:, col]])
        X_test[:, col]  = np.array([mapping.get(v, 0) for v in X_test[:, col]])

    #Normalizing numerical features
    X_train = X_train.astype(float)
    X_test  = X_test.astype(float)

    for col in continuous_cols:
        mean = X_train[:, col].mean()
        std  = X_train[:, col].std()
        if std == 0:
            std = 1.0

        X_train[:, col] = (X_train[:, col] - mean) / std
        X_test[:, col]  = (X_test[:, col]  - mean) / std
        

    return X_train, y_train, X_test, y_test
    


def z_normalize(X_train, X_test, feature_indices):
    """
    Apply z-score normalization to specified features.
    
    Parameters:
    -----------
    X_train, X_test : numpy arrays
    feature_indices : list of indices for numerical features
    
    Returns:
    --------
    X_train_normalized, X_test_normalized : numpy arrays
    """
    # TODO: Implement z-normalization
    X_train_normalized = X_train.copy().astype(float)
    X_test_normalized = X_test.copy().astype(float)

    for col in feature_indices:
        mean = X_train_normalized[:, col].mean()
        std = X_train_normalized[:, col].std()

        # Avoid division by zero
        if std == 0:
            std = 1.0

        X_train_normalized[:, col] = (X_train_normalized[:, col] - mean) / std
        X_test_normalized[:, col]  = (X_test_normalized[:, col]  - mean) / std

    return X_train_normalized, X_test_normalized

    



## Task 2: Implement k-NN Classifier
Implement k-NN with L2 (Euclidean) distance:
$$\mathcal{D}_{L2}(\mathbf{a}, \mathbf{b}) = \sqrt{\sum_i (a_i - b_i)^2}$$

For **categorical attributes**, use:
- Distance = 1 if values are different
- Distance = 0 if values are the same


In [ ]:
def l2_distance(a, b):
    """
    Compute L2 (Euclidean) distance between two vectors.
    
    Parameters:
    -----------
    a, b : numpy arrays of same shape
    
    Returns:
    --------
    distance : float
    """
    # TODO: Implement L2 distance
    return np.sqrt(np.sum((a - b) ** 2))
    pass


def knn_predict(X_train, y_train, X_test, k):
    """
    Predict labels for test data using k-NN.
    
    Parameters:
    -----------
    X_train : numpy array of shape (n_train, n_features)
    y_train : numpy array of shape (n_train,)
    X_test : numpy array of shape (n_test, n_features)
    k : int, number of neighbors
    
    Returns:
    --------
    predictions : numpy array of shape (n_test,)
    """
    # TODO: Implement k-NN prediction
    # For each test sample:
    #   1. Compute distance to all training samples
    #   2. Find k nearest neighbors
    #   3. Predict using majority voting
    
    
    pass


def compute_accuracy(y_true, y_pred):
    """
    Compute classification accuracy.
    
    Returns:
    --------
    accuracy : float (between 0 and 1)
    """
    # TODO: Implement accuracy computation
    pass


## Task 3: Evaluate on Lenses Dataset
Test your k-NN implementation on the Lenses dataset for different values of k.


In [ ]:
# TODO: Evaluate k-NN on Lenses dataset
# Try different values of k (e.g., 1, 3, 5, 7)

# k_values = [1, 3, 5, 7]
# lenses_results = []
# 
# for k in k_values:
#     predictions = knn_predict(X_train_lenses, y_train_lenses, X_test_lenses, k)
#     accuracy = compute_accuracy(y_test_lenses, predictions)
#     lenses_results.append((k, accuracy))
#     print(f"k={k}: Accuracy = {accuracy:.4f}")


## Task 4: Evaluate on Credit Approval Dataset
First preprocess the data, then evaluate k-NN.


In [ ]:
# TODO: Preprocess Credit Approval data
# X_train_credit, y_train_credit, X_test_credit, y_test_credit = preprocess_credit_data(
#     DATA_PATH + "crx.data.training",
#     DATA_PATH + "crx.data.testing"
# )


In [ ]:
# TODO: Evaluate k-NN on Credit Approval dataset
# k_values = [1, 3, 5, 7]
# credit_results = []
# 
# for k in k_values:
#     predictions = knn_predict(X_train_credit, y_train_credit, X_test_credit, k)
#     accuracy = compute_accuracy(y_test_credit, predictions)
#     credit_results.append((k, accuracy))
#     print(f"k={k}: Accuracy = {accuracy:.4f}")


## Summary and Discussion

### Results Table

| Dataset | k=1 | k=3 | k=5 | k=7 |
|---------|-----|-----|-----|-----|
| Lenses | ? | ? | ? | ? |
| Credit Approval | ? | ? | ? | ? |

### Discussion
*Answer these questions:*
1. Which value of k works best for each dataset? Why do you think that is?
2. How did preprocessing affect your results on the Credit Approval dataset?
3. What are the trade-offs of using different values of k?
4. What did you learn from this exercise?
